In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.data.market_loader import MarketLoader

from src.curves.curve_snapshot import CurveSnapshot
from src.curves.bootstrap.bootstrap_engine import BootstrapCurveEngine
from src.curves.projection_curve import ProjectionCurve
from src.curves.zero_curve import ZeroCurve

from src.instruments.instrument_builder import InstrumentBuilder

from src.trades.interest_rate_swap import InterestRateSwap

from src.pricing.swap_pricer import SwapPricer

from src.risk.exposure_engine import ExposureEngine

In [2]:
# downloading market curves
market_loader = MarketLoader()
market_curves = market_loader.market_loader_pipeline()

# downloading swap curves
swap_loader = MarketLoader()
swap_curves = swap_loader.swap_loader_pipeline()

treasury curve dataset already downloaded..
sofr curve dataset already downloaded..
futures curve dataset already downloaded..
estr curve dataset already downloaded..
usd_ois curve dataset already downloaded..
eur_ois curve dataset already downloaded..


In [3]:
### create curve snapshots
# SOFR snapshot
sofr_df = market_curves['sofr']

latest_date = sofr_df.index[-1]
latest_sofr_curve = sofr_df.iloc[-1]

sofr_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'sofr',
    as_of_date = latest_date,
    curve_row = latest_sofr_curve
)

# Futures snapshot
future_df = market_curves['futures']
latest_futures_curve = future_df.iloc[-1]

futures_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'futures',
    as_of_date = latest_date,
    curve_row = latest_futures_curve
)

# OIS snapshot
ois_df = swap_curves['usd_ois']

latest_swap_date = ois_df.index[-1]
latest_swap_curve = ois_df.iloc[-1]

ois_snapshot = CurveSnapshot.snap_from_df_row(
    curve_name = 'usd_ois',
    as_of_date = latest_swap_date,
    curve_row = latest_swap_curve
)

In [4]:
### create instruments from curve snapshot
# deposits
deposit_instruments = InstrumentBuilder.build_deposit_instruments(snapshot = sofr_snapshot)

# futures
future_instruments = InstrumentBuilder.build_future_instruments(snapshot = futures_snapshot)

# ois
ois_instruments = InstrumentBuilder.build_ois_instruments(snapshot = ois_snapshot)

### discount, projection and zero curve builder
# bootstrapping engine for generating the discount curve
all_instruments = deposit_instruments + future_instruments + ois_instruments

engine = BootstrapCurveEngine()

discount_curve = engine.bootstrap(
    snapshot = sofr_snapshot,
    instruments = all_instruments
)

# projection curve
projection_curve = ProjectionCurve(discount_curve = discount_curve)

# zero curve
zero_curve = ZeroCurve(discount_curve = discount_curve)

In [5]:
# sample IRS trade objects
IR_swap = InterestRateSwap(
    notional = 1_000_000,
    maturity = 3.0,
    fixed_rate = 3.40,
    pay_fixed = True
)

In [6]:
### exposure risk analytics
# pricer
pricer = SwapPricer(
    discount_curve = discount_curve,
    projection_curve = projection_curve
)

# exposure engine
exposure_engine = ExposureEngine(pricer = pricer)

# exposure profile
profile = exposure_engine.exposure_profile(
    swap = IR_swap,
    steps_per_year = 4
)

profile

,Time,PV,Exposure
0,0.00,7858.414276,7858.414276
1,0.25,-1061.295399,0.000000
2,0.50,6697.621395,6697.621395
3,0.75,-2770.287739,0.000000
4,1.00,4136.297444,4136.297444
5,1.25,-4581.910960,0.000000
6,1.50,2777.955866,2777.955866
7,1.75,-5940.252539,0.000000
8,2.00,1123.195202,1123.195202
9,2.25,-6845.236659,0.000000


In [7]:
# expected exposure
expected_exposure = exposure_engine.expected_exposure(
    swap = IR_swap,
    steps_per_year = 4
)

expected_exposure

,Time,EE,ENE
0,0.00,7858.414276,0.000000
1,0.25,0.000000,1061.295399
2,0.50,6697.621395,0.000000
3,0.75,0.000000,2770.287739
4,1.00,4136.297444,0.000000
5,1.25,0.000000,4581.910960
6,1.50,2777.955866,0.000000
7,1.75,0.000000,5940.252539
8,2.00,1123.195202,0.000000
9,2.25,0.000000,6845.236659


In [8]:
# exposure report
exposure_engine.exposure_report(
    swap = IR_swap,
    steps_per_year = 4
)

,Metric,Value
0,Current Exposure,7858.414276
1,Max Exposure,7858.414276
2,Average Exposure,1791.580394
